# Exercise 2: Boundary agreement

*Neural Morphology · version 0.1.0*

> For a Turkish word, obtain morph boundaries (from the TurkicNLP segmenter or Zemberek) and token boundaries (from a tokeniser), both as character offsets. Count two different errors separately: a *token that spans a morph boundary* (a morph boundary falls inside a token), and a *morph split by a token boundary* (a token boundary falls inside a morph). Apply the script to the 100 most frequent nouns of a corpus and report both rates for each tokeniser.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

The 100 most frequent noun forms (lowercased) come from the Turkish IMST training treebank. The morph boundaries come from TurkicNLP's `MorphemeTokenizer`, which in this run uses only its Apertium (HFST) backend. They are therefore the segmenter's boundaries, not gold boundaries; check a sample by hand before reporting them as such. A word counts only if both segmentations reproduce it exactly, so that the character offsets are comparable.

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
from turkicnlp.processors.morpheme_tokenizer import MorphemeTokenizer

seg = MorphemeTokenizer(lang="tur")
seg.load()
print("segmenter backends: hfst", seg._hfst_available, "| neural", seg._neural_available)   # private flags in 0.5.0

train = ud_sentences("UD_Turkish-IMST", "tr_imst-ud-train.conllu")
nouns = collections.Counter(f.lower() for _, ws in train for f, _, upos, _ in ws if upos == "NOUN" and f.isalpha())
top = [w for w, _ in nouns.most_common(100)]

def boundaries(parts):
    """Internal boundary offsets of a segmentation, e.g. ['ev', 'ler'] -> {2}."""
    out, i = set(), 0
    for p in parts[:-1]:
        i += len(p); out.add(i)
    return out

def spans(parts):
    out, i = [], 0
    for p in parts:
        out.append((i, i + len(p))); i += len(p)
    return out

morph_seg = {}
for w in top:
    s = seg.segment(w).segments
    if "".join(s) == w:
        morph_seg[w] = s
print(f"{len(top)} nouns; {len(morph_seg)} with a segmentation that reproduces the word; "
      f"{sum(len(s) > 1 for s in morph_seg.values())} with at least one internal morph boundary")

for label, name in [("BERTurk 32K", "dbmdz/bert-base-turkish-cased"), ("Glot500 base", "cis-lmu/glot500-base"),
                    ("NLLB-200", "facebook/nllb-200-distilled-600M")]:
    try:
        tok = load_tokenizer(name)
    except Exception as exc:
        print(f"{label}: not available here ({type(exc).__name__})"); continue
    tok_spanning = tok_total = morph_split = morph_total = 0
    for w, mparts in morph_seg.items():
        tparts = pieces(tok, w)
        if "".join(tparts) != w:
            continue
        mb, tb = boundaries(mparts), boundaries(tparts)
        for a, b in spans(tparts):                    # a token spans a morph boundary
            tok_total += 1; tok_spanning += any(a < x < b for x in mb)
        for a, b in spans(mparts):                    # a morph is split by a token boundary
            morph_total += 1; morph_split += any(a < x < b for x in tb)
    print(f"{label}: tokens spanning a morph boundary {tok_spanning}/{tok_total} = {tok_spanning/tok_total:.1%}; "
          f"morphs split by a token boundary {morph_split}/{morph_total} = {morph_split/morph_total:.1%}")
print("examples:", {w: morph_seg[w] for w in top[:8]})

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026:

```
  ↓ Downloading cis-lmu/glot500-base backbone...
  ↓ Downloading cis-lmu/glot500-base backbone...
segmenter backends: hfst True | neural False
100 nouns; 100 with a segmentation that reproduces the word; 41 with at least one internal morph boundary
BERTurk 32K: not available here (OSError)
Glot500 base: tokens spanning a morph boundary 39/118 = 33.1%; morphs split by a token boundary 13/148 = 8.8%
NLLB-200: tokens spanning a morph boundary 39/122 = 32.0%; morphs split by a token boundary 14/148 = 9.5%
examples: {'zaman': ['zaman'], 'şey': ['şey'], 'gün': ['gün'], 'bilim': ['bilim'], 'ev': ['ev'], 'yerine': ['yerin', 'e'], 'çocuk': ['çocuk'], 'annem': ['anne', 'm']}
```

All 100 nouns get a segmentation that reproduces the word, but only 41 of them get an internal boundary: most frequent nouns are bare stems (*zaman*, *şey*, *gün*). The morph boundaries come from the Apertium backend of the segmenter and include errors: *yerine* 'instead of, to its place' is segmented *yerin-e* instead of *yer-i-n-e*, so the possessive and the buffer are merged into the stem.

| Tokeniser | Tokens spanning a morph boundary | Morphs split by a token boundary |
|---|---|---|
| Glot500 base | 39/118 = 33.1% | 13/148 = 8.8% |
| NLLB-200 | 39/122 = 32.0% | 14/148 = 9.5% |

The two error types behave very differently. A third of the tokens contain a morph boundary: frequent inflected forms such as *yerine* are stored as one token, so stem and suffix stay together. Fewer than one morph in ten is split by a token boundary: when the tokenisers cut, they mostly cut between morphs (*ev-ler*, *kitap-ları* with both tokenisers), but not always (*annem* becomes *an-nem* with Glot500 and *ann-em* with NLLB-200, both inside the stem *anne*). Both rates depend on the segmenter's boundaries; with checked boundaries they will change (Your turn, 1). BERTurk could not be loaded here.

## The recurring sentence

Three words of the book's recurring Kazakh sentence, segmented by the Kazakh `MorphemeTokenizer` and split by the two tokenisers.

In [ ]:
kseg = MorphemeTokenizer(lang="kaz")
kseg.load()
words = ["өңірдегі", "бейбітшілікті", "қолдайды"]          # from the recurring Kazakh sentence
toks = {}
for label, name in [("Glot500 base", "cis-lmu/glot500-base"), ("NLLB-200", "facebook/nllb-200-distilled-600M")]:
    try:
        toks[label] = load_tokenizer(name)
    except Exception as exc:
        print(f"{label}: not available here ({type(exc).__name__})")
for w in words:
    print(f"{w:<15} morphs {kseg.segment(w).segments}", *(f"| {k} {pieces(t, w)}" for k, t in toks.items()))

Recorded output (20 October 2026; TurkicNLP 0.5.0 with its Apertium backend only):

```
өңірдегі        morphs ['өңір', 'де', 'гі'] | Glot500 base ['өңір', 'дегі'] | NLLB-200 ['өңір', 'дегі']
бейбітшілікті   morphs ['бейбітшілік', 'ті'] | Glot500 base ['бей', 'біт', 'шілік', 'ті'] | NLLB-200 ['бей', 'біт', 'ші', 'лікті']
қолдайды        morphs ['қол', 'дай', 'ды'] | Glot500 base ['қол', 'дайды'] | NLLB-200 ['қол', 'дайды']
```

Both tokenisers keep *өңір* whole and put the locative and the attributive into one token *дегі*, a token that spans a morph boundary. Glot500 isolates the accusative *ті*; NLLB-200 splits the stem and merges the suffix into *лікті*. The segmenter is wrong on *қолдайды*: it chose the similative analysis *қол-дай-ды* 'like a hand (ACC)' instead of the verb *қолда-й-ды* 'supports'. The verb has a reading in the transducer, but the segmenter picks without context, so the morph boundaries used as the reference can themselves be errors (Your turn, 1).

## Your turn — extension

1. Check the segmenter's boundaries for the 100 nouns by hand and recompute both rates against your corrected boundaries.
2. Add BERTurk and XLM-R where the Hub is reachable.